In [1]:
pip install pandas wbgapi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import wbgapi as wb
from pathlib import Path

In [3]:
# =====================================================
# Config
# =====================================================

START_YEAR = 2010
END_YEAR = 2025

INDICATORS = {
    "gdp_per_capita": "NY.GDP.PCAP.CD",
    "real_gdp_growth": "NY.GDP.MKTP.KD.ZG",
    "deposit_interest_rate": "FR.INR.DPST",
    "inflation_rate": "FP.CPI.TOTL.ZG",
    "compulsory_education_years": "SE.COM.DURS",
    "population_65_plus_pct": "SP.POP.65UP.TO.ZS",
    "fertility_rate": "SP.DYN.TFRT.IN",
    "unemployment_rate": "SL.UEM.TOTL.ZS",
    "government_expenditure_pct_gdp": "GC.XPN.TOTL.GD.ZS",
    "government_revenue_pct_gdp": "GC.REV.XGRT.GD.ZS",
    "gdp_deflator_inflation": "NY.GDP.DEFL.KD.ZG"
}


In [14]:
countries = pd.DataFrame(list(wb.economy.list()))

# Remove agregados (World, OECD, etc.)
countries = countries[countries["aggregate"] == False]

country_codes = countries["id"].tolist()

print(f"{len(country_codes)} países encontrados.")

217 países encontrados.


In [13]:
df = wb.data.DataFrame(
    "NY.GDP.PCAP.CD",
    economy=country_codes,
    time=range(2020, 2026),
    labels=True
)

# transforma índice em coluna
df = df.reset_index()

# formato longo
df = df.melt(
    id_vars=["economy", "Country"],
    var_name="year",
    value_name="gdp_per_capita"
)

# limpa o ano
df["year"] = (
    df["year"]
    .str.replace("YR", "", regex=False)
    .astype(int)
)

print(df.head())

  economy       Country  year  gdp_per_capita
0     ZWE      Zimbabwe  2020     2059.674454
1     ZMB        Zambia  2020      951.644317
2     ZAF  South Africa  2020     5580.603831
3     YEM   Yemen, Rep.  2020             NaN
4     XKX        Kosovo  2020     4310.889638


In [ ]:
# PAÍSES
# =====================================================

countries = pd.DataFrame(list(wb.economy.list()))
countries = countries[countries["aggregate"] == False]

country_codes = countries["id"].tolist()

print(f"{len(country_codes)} países encontrados")

# INDICADORES
# =====================================================

INDICATORS = {
    "gdp_per_capita": "NY.GDP.PCAP.CD",
    "real_gdp_growth": "NY.GDP.MKTP.KD.ZG",
    "inflation_rate": "FP.CPI.TOTL.ZG",
    "compulsory_education_years": "SE.COM.DURS",
    "population_65_plus_pct": "SP.POP.65UP.TO.ZS",
    "fertility_rate": "SP.DYN.TFRT.IN",
    "unemployment_rate": "SL.UEM.TOTL.ZS",
    "government_expenditure_pct_gdp": "GC.XPN.TOTL.GD.ZS",
    "government_revenue_pct_gdp": "GC.REV.XGRT.GD.ZS",
    "gdp_deflator_inflation": "NY.GDP.DEFL.KD.ZG",
    "current_account_pct_gdp": "BN.CAB.XOKA.GD.ZS",
    "external_debt_pct_gni": "DT.DOD.DECT.GN.ZS",
    "age_dependency_ratio": "SP.POP.DPND",
    "gross_capital_formation_pct_gdp": "NE.GDI.TOTL.ZS",
    "total_reserves_months_imports": "FI.RES.TOTL.MO",
    "savings_pct_gdp": "NY.GNS.ICTR.ZS",
    "exports_pct_gdp": "NE.EXP.GNFS.ZS"
}

# FUNÇÃO DE DOWNLOAD
# =====================================================

def download_indicator(indicator_code, variable_name):

    print(f"Baixando {variable_name}...")

    df = wb.data.DataFrame(
        indicator_code,
        economy=country_codes,
        time=range(2010, 2024),
        labels=True
    )

    df = df.reset_index()

    df = df.melt(
        id_vars=["economy", "Country"],
        var_name="year",
        value_name=variable_name
    )

    df["year"] = (
        df["year"]
        .str.replace("YR", "", regex=False)
        .astype(int)
    )

    return df


# PRIMEIRO INDICADOR
# =====================================================

first_var = list(INDICATORS.keys())[0]
first_code = INDICATORS[first_var]

panel = download_indicator(first_code, first_var)

# MERGE DOS DEMAIS
# =====================================================

for var, code in list(INDICATORS.items())[1:]:

    temp = download_indicator(code, var)

    panel = panel.merge(
        temp[
            ["economy", "year", var]
        ],
        on=["economy", "year"],
        how="left"
    )

# RENOMEIA
# =====================================================

panel = panel.rename(columns={
    "economy": "country_code",
    "Country": "country"
})

# ORDENA
# =====================================================

panel = panel.sort_values(
    ["country", "year"]
).reset_index(drop=True)

# RESULTADO
# =====================================================

print(panel.head())

print("\nShape:")
print(panel.shape)

print("\nMissing Values:")
print(
    panel.isna()
    .mean()
    .sort_values(ascending=False)
)

# SALVA
# =====================================================

panel.to_csv(
    "wdi_fiscal_risk_panel.csv",
    index=False
)

print("\nArquivo salvo!")

217 países encontrados
Baixando gdp_per_capita...


KeyboardInterrupt: 

In [19]:
coverage = (
    panel
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values()
)

print(coverage)

country_code                       0.00
country                            0.00
year                               0.00
population_65_plus_pct             0.00
fertility_rate                     0.00
gdp_per_capita                     2.83
real_gdp_growth                    3.49
gdp_deflator_inflation             3.72
compulsory_education_years        10.76
unemployment_rate                 13.96
inflation_rate                    14.78
government_revenue_pct_gdp        38.51
government_expenditure_pct_gdp    39.47
deposit_interest_rate             41.84
dtype: float64


In [20]:
summary = pd.DataFrame({
    "missing_pct": panel.isna().mean()*100,
    "non_missing": panel.notna().sum()
})

print(summary)

                                missing_pct  non_missing
country_code                       0.000000         3038
country                            0.000000         3038
year                               0.000000         3038
gdp_per_capita                     2.830810         2952
real_gdp_growth                    3.489138         2932
deposit_interest_rate             41.836735         1767
inflation_rate                    14.779460         2589
compulsory_education_years        10.763660         2711
population_65_plus_pct             0.000000         3038
fertility_rate                     0.000000         3038
unemployment_rate                 13.956550         2614
government_expenditure_pct_gdp    39.466754         1839
government_revenue_pct_gdp        38.512179         1868
gdp_deflator_inflation             3.719552         2925
